# Preliminary BRB-seq expression PCA

This notebook only reads the deep FastReadCounter matrices, maps barcodes to sample IDs, applies DESeq2's variance-stabilizing transform, and makes separate gill and liver PCA plots. No differential-expression model is fitted here.

The compiled sample metadata are used for labels and annotations. `2024_DJR_196` is intentionally retained with unknown sex so its position can be inspected in the gill PCA.

In [1]:
library(DESeq2)
library(ggplot2)

# Edit these paths only if the repository/output location changes.
repo_root <- "/scratch/ztobias/Gacu_BC_txmics"
count_root <- file.path(repo_root, "outputs", "matrix")
metadata_file <- file.path(repo_root, "metadata", "derived", "sample_tracking.tsv")
barcode_file <- file.path(repo_root, "metadata", "brb_sample_barcodes.tsv")

tracking <- read.delim(metadata_file, header = TRUE, sep = "\t",
  stringsAsFactors = FALSE, check.names = FALSE)
brb_barcodes <- read.delim(barcode_file, header = TRUE, sep = "\t",
  stringsAsFactors = FALSE, check.names = FALSE)

tracking <- tracking[grepl("^(2024|2025)_DJR_", tracking$sample_id), ]
brb_barcodes <- brb_barcodes[grepl("^(2024|2025)_DJR_", brb_barcodes$sample_id), ]
head(tracking[, c("sample_id", "population", "ecotype", "field_sex", "true_sex")])

Loading required package: S4Vectors

Warning message:
“package ‘S4Vectors’ was built under R version 4.1.3”
Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Warning message:
“package ‘IRanges’ was built under R version 4.1.3”
Loading required packag

,sample_id,population,ecotype,field_sex,true_sex
,<chr>,<chr>,<chr>,<chr>,<chr>
1,2024_DJR_001,Priest,limnetic,M,M
2,2024_DJR_002,Priest,limnetic,F,F
3,2024_DJR_003,Priest,benthic,F,F
4,2024_DJR_004,Priest,limnetic,M,M
5,2024_DJR_005,Priest,limnetic,F,F
6,2024_DJR_006,Priest,benthic,M,M


## Read and map the four deep count matrices

Each year/tissue run has its own matrix. The 2025 gill and liver matrices already represent the independent tissue run plus the combined lane, because those FASTQs were concatenated before STAR.

In [2]:
read_count_matrix <- function(path) {
  if (!file.exists(path)) stop("Missing count matrix: ", path)
  x <- read.delim(path, header = TRUE, row.names = 1,
    check.names = FALSE, stringsAsFactors = FALSE)
  x <- as.matrix(x)
  storage.mode(x) <- "numeric"
  x
}

matrix_file <- function(tissue, year) {
  file.path(count_root, paste0(tissue, "_", year,
    "_deep_fastreadcounter"), "counts.txt")
}

map_matrix_to_samples <- function(x, tissue, year) {
  map <- brb_barcodes[brb_barcodes$tissue == tissue &
    brb_barcodes$year == as.integer(year), c("sample_id", "barcode")]
  map <- map[!duplicated(map$barcode), ]
  sample_names <- map$sample_id[match(colnames(x), map$barcode)]
  if (anyNA(sample_names)) {
    cat("Unmapped barcodes for ", tissue, " ", year, ":\n")
    print(colnames(x)[is.na(sample_names)])
  }
  keep <- !is.na(sample_names)
  x <- x[, keep, drop = FALSE]
  colnames(x) <- sample_names[keep]
  x
}

read_tissue_counts <- function(tissue) {
  mats <- lapply(c("2024", "2025"), function(year) {
    x <- read_count_matrix(matrix_file(tissue, year))
    map_matrix_to_samples(x, tissue, year)
  })
  genes <- Reduce(intersect, lapply(mats, rownames))
  mats <- lapply(mats, function(x) x[genes, , drop = FALSE])
  out <- do.call(cbind, mats)
  if (anyDuplicated(colnames(out))) stop("Duplicate sample IDs in ", tissue, " matrix")
  out
}

gill_counts <- read_tissue_counts("gill")
liver_counts <- read_tissue_counts("liver")
dim(gill_counts)
dim(liver_counts)

Unmapped barcodes for  gill   2024 :
 [1] "ACGATGCGACTACG" "ACTTAACCATACCA" "CCGCAATATGGCAA" "CTACTCAGTGTCGC"
 [5] "CTATGCTAGCGAGC" "TATGCCATGCCTAC" "TCGACTGCATGCGA" "TCTTACAGCCAGCC"
 [9] "TTACATCTAGTCCA" "Unknown"       
Unmapped barcodes for  gill   2025 :
  [1] "AACAACGCCGGCCA" "AACAACTCGCACGG" "AACAAGGTGTCCAA" "AACCAGAGAATGGC"
  [5] "AACCGTGACATTAC" "AACCTGAGGCAAGA" "AACGGTACTGAAGA" "AACGTTAAGTCCGA"
  [9] "AAGAACGGCCTGGC" "AAGCATTCTGGCCG" "AAGCCGCACACAAG" "AAGCCGTGATTCCG"
 [13] "AAGGAAGTGCGCCA" "AAGGCGCTTCAACG" "AAGTGAATACCAAC" "AATCCTGGTACACG"
 [17] "AATGCCTTGTAAGG" "AATGGAACCGTGAG" "AATTATACCGCTCA" "ACAATACCGCCAAG"
 [21] "ACACACGAGCAGCG" "ACACATATGACAGC" "ACACGACAGCTCAC" "ACACTACTCGTGCA"
 [25] "ACAGATGCGTGCGC" "ACAGCAGTATAGCA" "ACAGCTGCTGTACA" "ACATAGTCATCTGA"
 [29] "ACATGGAGGAGAAC" "ACCGGCCAATAACA" "ACGATGCTCATCAC" "ACGGTTCAACATAA"
 [33] "ACGTACTAGAGCGA" "ACGTCAGCATCGAC" "ACTAGACGCACACA" "ACTCCATATTCGGC"
 [37] "ACTCTGCGCTCGAC" "ACTGAGTCTAGAGA" "ACTGGCCTAGTGGA" "ATACGCGACGCTCA"
 

[1] 28430   636

[1] 28430   639

In [3]:
make_coldata <- function(counts) {
  md <- tracking[match(colnames(counts), tracking$sample_id), , drop = FALSE]
  if (anyNA(md$sample_id)) stop("Count matrix sample missing from compiled metadata")
  rownames(md) <- md$sample_id
  md
}

gill_metadata <- make_coldata(gill_counts)
liver_metadata <- make_coldata(liver_counts)

# Unknown sex is retained as NA. Do not replace it before inspecting PCA.
gill_metadata[gill_metadata$sample_id == "2024_DJR_196",
  c("sample_id", "field_sex", "true_sex", "population", "ecotype")]

table(gill_metadata$year, useNA = "ifany")
table(liver_metadata$year, useNA = "ifany")

,sample_id,field_sex,true_sex,population,ecotype
,<chr>,<chr>,<chr>,<chr>,<chr>
2024_DJR_196,2024_DJR_196,?,,Boot,lake



2024 2025 
 375  261 


2024 2025 
 375  264 

## DESeq2 variance stabilization

This uses the same basic logic as the 2024 analysis: remove genes with no useful counts, build a DESeq2 object, and use a blind VST for exploratory PCA. No `DESeq()` call or statistical test is run.

In [4]:
# Inspect the matrices before giving them to DESeq2.
count_diagnostics <- function(counts, label) {
  cat("\n--- ", label, " ---\n")
  cat("dimensions: ", nrow(counts), " genes x ", ncol(counts), " samples\n")
  cat("NA values: ", sum(is.na(counts)), "\n")
  cat("negative values: ", sum(counts < 0, na.rm = TRUE), "\n")
  cat("all-zero genes: ", sum(rowSums(counts) == 0), "\n")
  cat("all-zero samples: ", sum(colSums(counts) == 0), "\n")
  sample_summary <- data.frame(
    sample = colnames(counts),
    total_counts = colSums(counts),
    detected_genes = colSums(counts > 0),
    zero_genes = colSums(counts == 0)
  )
  print(summary(sample_summary$total_counts))
  print(summary(sample_summary$detected_genes))
  print(sample_summary[order(sample_summary$total_counts), ])
  invisible(sample_summary)
}

gill_count_diagnostics <- count_diagnostics(gill_counts, "gill")
liver_count_diagnostics <- count_diagnostics(liver_counts, "liver")


---  gill  ---
dimensions:  28430  genes x  636  samples
NA values:  0 
negative values:  0 
all-zero genes:  3051 
all-zero samples:  0 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
     71 2899443 3287208 3254385 3691501 6659615 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
     57   16345   16826   17078   18422   20255 
                   sample total_counts detected_genes zero_genes
2025_DJR_415 2025_DJR_415           71             57      28373
2025_DJR_423 2025_DJR_423           74             63      28367
2025_DJR_419 2025_DJR_419           75             61      28369
2025_DJR_421 2025_DJR_421           83             68      28362
2025_DJR_246 2025_DJR_246           99             76      28354
2025_DJR_413 2025_DJR_413          101             78      28352
2025_DJR_416 2025_DJR_416          113             76      28354
2025_DJR_267 2025_DJR_267       242373          12627      15803
2025_DJR_286 2025_DJR_286       420718          14500      13930
2025_DJR_266 20


---  liver  ---
dimensions:  28430  genes x  639  samples
NA values:  0 
negative values:  0 
all-zero genes:  3089 
all-zero samples:  0 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 157487 1953006 2166776 2386297 2892811 7918287 
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   7412   12805   13301   13271   13886   17457 
                   sample total_counts detected_genes zero_genes
2024_DJR_104 2024_DJR_104       157487           7412      21018
2024_DJR_203 2024_DJR_203       171516           7574      20856
2024_DJR_094 2024_DJR_094       361246           8509      19921
2024_DJR_039 2024_DJR_039       448551           8615      19815
2024_DJR_105 2024_DJR_105       475402          10319      18111
2024_DJR_295 2024_DJR_295       606216          11538      16892
2024_DJR_085 2024_DJR_085       717313          10351      18079
2024_DJR_097 2024_DJR_097       754995          11313      17117
2024_DJR_407 2024_DJR_407       802341           9697      18733
2024_DJR_009 2

In [ ]:
make_vsd <- function(counts, metadata) {
  keep <- rowSums(counts >= 10) >= max(2, ceiling(ncol(counts) * 0.10))
  dds <- DESeqDataSetFromMatrix(
    countData = round(counts[keep, , drop = FALSE]),
    colData = metadata,
    design = ~ 1
  )
  list(
    dds = dds,
    vsd = vst(dds, blind = TRUE),
    genes_kept = sum(keep)
  )
}

gill_vsd <- make_vsd(gill_counts, gill_metadata)
liver_vsd <- make_vsd(liver_counts, liver_metadata)
c(gill = gill_vsd$genes_kept, liver = liver_vsd$genes_kept)

In [ ]:
make_pca_data <- function(vsd_obj, metadata) {
  pca <- prcomp(t(assay(vsd_obj)), center = TRUE, scale. = FALSE)
  variance <- 100 * pca$sdev^2 / sum(pca$sdev^2)
  out <- cbind(metadata,
    PC1 = pca$x[, 1], PC2 = pca$x[, 2])
  attr(out, "variance") <- variance
  out
}

gill_pca_data <- make_pca_data(gill_vsd$vsd, gill_metadata)
liver_pca_data <- make_pca_data(liver_vsd$vsd, liver_metadata)

plot_pca <- function(pca_data, title) {
  variance <- attr(pca_data, "variance")
  ggplot(pca_data, aes(PC1, PC2, color = ecotype, shape = true_sex)) +
    geom_point(size = 2.2, na.rm = FALSE) +
    geom_text(
      data = pca_data[pca_data$sample_id == "2024_DJR_196", , drop = FALSE],
      aes(label = sample_id), nudge_y = 0.02, na.rm = TRUE, show.legend = FALSE
    ) +
    facet_wrap(~ population) +
    labs(
      title = title,
      subtitle = "Exploratory VST PCA; no DE model fitted",
      x = paste0("PC1 (", round(variance[1], 1), "%)"),
      y = paste0("PC2 (", round(variance[2], 1), "%)"),
      shape = "WGS/compiled sex"
    ) +
    theme_classic()
}

plot_pca(gill_pca_data, "Gill BRB-seq")
plot_pca(liver_pca_data, "Liver BRB-seq")

### Interpretation note

If `2024_DJR_196` clusters clearly with one sex in the gill PCA, that is useful evidence for follow-up, but it is not by itself a genetic-sex assignment. Population, ecotype, batch, sequencing depth, and sample-quality structure can also drive PCA separation.